# Bài 7 – Trang 27: Nhận dạng chữ số viết tay

**6451071011 - Lê Nguyễn Anh Dự**

Bài dùng bộ `digits` tích hợp trong scikit-learn (ảnh 8×8, mức sáng 0–16), cây quyết định theo entropy và hai ảnh viết tay được cung cấp. Chạy các ô theo thứ tự trong Google Colab.


In [ ]:
import os
import sys
import subprocess
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image
from IPython.display import display
from sklearn import datasets, metrics
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.model_selection import train_test_split

STUDENT = '6451071011 - Lê Nguyễn Anh Dự'
IMAGE_IDS = {'Ảnh 1': '1BH2qwN62GKZIHcF0kUi0N13YG2eex3wx',
             'Ảnh 2': '1KU83z5OU2j69YBPI7uMPcLPSUxidd6Ia'}

def heading(number, title):
    print(STUDENT)
    print(f'Bước {number}. {title}')

def ensure_image(label, file_id):
    filename = 'bai7_anh_1.jpg' if label == 'Ảnh 1' else 'bai7_anh_2.jpg'
    if not os.path.exists(filename):
        try:
            import gdown
        except ImportError:
            subprocess.check_call([sys.executable, '-m', 'pip', '-q', 'install', 'gdown'])
            import gdown
        gdown.download(id=file_id, output=filename, quiet=False)
    if not os.path.exists(filename):
        raise FileNotFoundError(f'Không tải được {filename}; hãy tải ảnh lên Colab.')
    return filename


## Bước 1. Nạp dữ liệu chữ số và xem mẫu

In [ ]:
heading(1, 'Nạp bộ digits và hiển thị mẫu 0, 1, 2, 3')
digits = datasets.load_digits()
print('Số mẫu:', len(digits.images), '| Kích thước ảnh:', digits.images.shape[1:])
fig, axes = plt.subplots(1, 4, figsize=(8, 2.5))
for ax, label in zip(axes, range(4)):
    idx = np.flatnonzero(digits.target == label)[0]
    ax.imshow(digits.images[idx], cmap='gray_r', vmin=0, vmax=16, interpolation='nearest')
    ax.set_title(f'Training: {label}')
    ax.axis('off')
plt.tight_layout(); plt.show()


## Bước 2. Chuyển ảnh 8×8 thành vector 64 thuộc tính

In [ ]:
heading(2, 'Làm phẳng dữ liệu ảnh')
X = digits.images.reshape((len(digits.images), -1))
y = digits.target
print('X:', X.shape, '| y:', y.shape)
print('Miền giá trị pixel:', X.min(), 'đến', X.max())


## Bước 3. Tạo cây quyết định và chia dữ liệu 80/20

In [ ]:
heading(3, 'Huấn luyện cây ID3/entropy')
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, shuffle=False)
classifier = DecisionTreeClassifier(criterion='entropy', random_state=33)
classifier.fit(X_train, y_train)
print('Train:', X_train.shape, '| Test:', X_test.shape)
print('Số nút:', classifier.tree_.node_count, '| Độ sâu:', classifier.get_depth())
plt.figure(figsize=(20, 8))
plot_tree(classifier, max_depth=2, filled=True, rounded=True,
          feature_names=[f'pixel_{i}' for i in range(64)], fontsize=8)
plt.title('Ba tầng đầu của cây entropy')
plt.tight_layout(); plt.show()


## Bước 4. Dự đoán và đánh giá trên tập kiểm thử

`criterion="entropy"` dùng information gain như hướng dẫn, nhưng cây scikit-learn chia các giá trị số theo ngưỡng nên là cách mô phỏng ID3.


In [ ]:
heading(4, 'Dự đoán và ma trận nhầm lẫn')
predicted = classifier.predict(X_test)
fig, axes = plt.subplots(1, 4, figsize=(9, 2.5))
for ax, image, pred in zip(axes, X_test[:4].reshape(-1, 8, 8), predicted[:4]):
    ax.imshow(image, cmap='gray_r', vmin=0, vmax=16, interpolation='nearest')
    ax.set_title(f'Dự đoán: {pred}')
    ax.axis('off')
plt.tight_layout(); plt.show()
print(metrics.classification_report(y_test, predicted, zero_division=0))
print('Accuracy:', round(metrics.accuracy_score(y_test, predicted), 4))
cm = metrics.confusion_matrix(y_test, predicted, labels=np.arange(10))
plt.figure(figsize=(9, 7))
sns.heatmap(cm, annot=True, fmt='d', cmap='viridis',
            xticklabels=np.arange(10), yticklabels=np.arange(10))
plt.xlabel('Dự đoán'); plt.ylabel('Thực tế'); plt.title('Ma trận nhầm lẫn – digits')
plt.tight_layout(); plt.show()


## Bước 5. Đọc và tiền xử lý hai ảnh viết tay

Hai ảnh có nét đỏ trên nền sáng. Để gần với ảnh huấn luyện, ta tách nét đỏ, cắt vùng chứa chữ số, đặt vào khung vuông có lề, rồi giảm về 8×8 và đổi thang sáng thành 0–16. Giá trị `?` không liên quan ở bài này. Kết quả nhận dạng ảnh chụp có thể khác nhãn thật vì miền ảnh khác bộ `digits`.


In [ ]:
heading(5, 'Tách nét đỏ và đưa về ảnh 8×8')

def preprocess_red_digit(filename):
    original = Image.open(filename).convert('RGB')
    rgb = np.asarray(original).astype(np.float32)
    red = rgb[..., 0] - (rgb[..., 1] + rgb[..., 2]) / 2
    # Màu đỏ trong ảnh rõ hơn nền xám; ngưỡng thấp giữ nét mảnh.
    strength = np.clip((red - 12) / 65, 0, 1)
    mask = strength > 0.20
    if not mask.any():
        raise ValueError(f'Không tìm thấy nét đỏ trong {filename}.')
    yy, xx = np.where(mask)
    x0, x1 = xx.min(), xx.max() + 1
    y0, y1 = yy.min(), yy.max() + 1
    crop = strength[y0:y1, x0:x1]
    h, w = crop.shape
    side = max(h, w)
    canvas = np.zeros((side, side), dtype=np.float32)
    top, left = (side-h)//2, (side-w)//2
    canvas[top:top+h, left:left+w] = crop
    # Thu nét vào vùng giữa, để khoảng trống giống bộ digits.
    small = Image.fromarray(np.uint8(canvas * 255)).resize((6, 6), Image.Resampling.LANCZOS)
    image8 = np.zeros((8, 8), dtype=np.float32)
    image8[1:7, 1:7] = np.asarray(small).astype(np.float32) / 255 * 16
    return original, image8

processed = {}
fig, axes = plt.subplots(2, 2, figsize=(7, 7))
for row, (label, file_id) in enumerate(IMAGE_IDS.items()):
    original, image8 = preprocess_red_digit(ensure_image(label, file_id))
    processed[label] = image8
    axes[row, 0].imshow(original); axes[row, 0].set_title(f'{label}: ảnh gốc')
    axes[row, 1].imshow(image8, cmap='gray_r', vmin=0, vmax=16,
                        interpolation='nearest'); axes[row, 1].set_title(f'{label}: sau xử lý 8×8')
    for ax in axes[row]: ax.axis('off')
plt.tight_layout(); plt.show()


## Bước 6. Nhận dạng hai ảnh viết tay

In [ ]:
heading(6, 'Dự đoán ảnh 1 và ảnh 2')
for label, image8 in processed.items():
    vector = image8.reshape(1, 64)
    prediction = classifier.predict(vector)[0]
    print(f'{label}: dự đoán số {prediction}')
print('Nhãn nhìn bằng mắt: ảnh 1 là 1, ảnh 2 là 2. Nếu dự đoán khác, cần chỉnh cách tách nền/căn giữa hoặc bổ sung ảnh chụp tương tự vào tập huấn luyện.')
